# Every branch, or say why not

A trace runs your function once, so it sees one path. If your code
branches on the data, the formula you get is the formula for the
branch your input happened to take. This notebook shows the danger,
and the fix.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import sympy
from skverify.testing import check_formula
from skverify.explore import explore

v = sympy.IndexedBase("v")
i = sympy.Symbol("i", integer=True)

## The danger

Here is a function with two branches, and a claim about it that is
half right. On inputs with a positive sum it doubles. On the rest it
triples. Our spec says it always doubles.

In [2]:
def f(x):
    if x.sum() > 0:
        return x * 2.0
    return x * 3.0

x = sympy.IndexedBase("x")
verdict = check_formula(f, (np.array([1.0, 2.0]),), 2 * x[i], indices=(i,))
print(verdict.message())

verdict: exact (at shape (2,))
  2/2 entries agree


The check passes, and it is not lying: on the path where the sum is
positive, the code really does compute 2x. The message says which
path. But our test input had a positive sum, so the other branch was
never looked at, and the spec is wrong there. Every test suite in the
world has this blind spot: you check the inputs you thought of.

## The fix

`explore=True` closes it. The trace records the branch condition
(the sum is positive). The explorer negates it and asks the Z3
solver for an input on the other side. The solver produces one, the
function runs again on that input, and the spec is checked on the
second branch too.

In [3]:
verdict = check_formula(f, (np.array([1.0, 2.0]),), 2 * x[i],
                        indices=(i,), explore=True)
print(verdict.message())

verdict: differs (at shape (2,))
  your spec:  2*x[0]
  the code:   3.0*x[0]
  counterexample:
      x[0] = 21/10
      spec value = 4.2
      code value = 6.300000000000001
  first disagreement at entry (0,)
  on the path where: Sum(x[j], (j, 0, 1)) <= 0


Caught. The spec fails on the branch where the sum is not positive,
and the message names that branch. No amount of staring at our
original test input would have found this, because the bug is not at
any input we ran. It is on a path we never took.

## What exploration looks like

`explore` on its own shows the map: which paths exist, which were
visited, and what happened to the regions in between. Three
outcomes are possible for a region, and all three are stated, never
guessed:

* visited: the solver produced an input, the function ran on it
* proven infeasible: the solver showed no input reaches it
* undecided: neither, named in the result, and coverage is not claimed

In [4]:
def clip_like(x):
    if x[0] > 1.0:
        return x * 2.0
    if x[0] > 0.0:
        return x + 1.0
    return x

r = explore(clip_like, (np.array([1.0, 2.0]),))
print(r.summary())
for p in r.paths:
    print("  path:", p.condition)

3 path(s) explored, 1 region(s) proven infeasible -- coverage proven
  path: (x[0] <= 1.0, x[0] > 0.0)
  path: (x[0] <= 0.0, x[0] <= 1.0)
  path: (x[0] > 1.0,)


Three branches, three paths, and the fourth combination (x above 1
and below 0 at the same time) was proven impossible rather than
silently skipped.

A subtler case: `np.sinc` has a special value at zero. That branch
is a single point, so no test input ever lands on it by luck. The
solver computes it.

In [5]:
r = explore(lambda a: np.sinc(a), (np.array([0.7, -1.2, 2.5]),))
print(r.summary())
print("paths found:", len(r.paths))

8 path(s) explored -- coverage proven


paths found: 8


Eight paths: every combination of which entries are zero. The
degenerate cases that break numerical code in practice are exactly
these measure-zero branches, and they are the ones random testing
can never reach.

## The honest edges

Two limits, stated plainly. First, the proof is for the traced input
shape: all values of a length-2 array is a theorem, length 3 is a
new run. Second, functions whose branch count explodes (sorting 6
elements has hundreds of orderings) hit a path budget, and the
result says "stopped at the cap" instead of claiming coverage.

Measured over every public numpy function the tracer lifts, 274 of
293 get coverage proven, and the rest say exactly which of the two
limits they hit. The boards in `coverage/` regenerate the number.